# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [5]:
import sys
sys.path.append("..")

from common_config import llm_connect

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [9]:
llm = llm_connect(model="gpt-5.4-mini", temperature=0, max_tokens=2086)

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에 너무 딱 맞게 배워서, 새로운 데이터에 대해서는 성능이 떨어지는 현상**입니다.

### 쉽게 말하면
- **학습 데이터**: 문제집을 너무 외워버림
- **테스트 데이터**: 처음 보는 문제를 못 풂

즉, 모델이 데이터의 **진짜 패턴**보다 **우연한 잡음이나 특수한 부분**까지 외워버린 상태예요.

### 특징
- 학습 데이터 성능은 매우 좋음
- 검증/테스트 데이터 성능은 떨어짐
- 모델이 너무 복잡할 때 자주 발생함

### 예시
시험 문제를 풀 때,
- 개념을 이해한 학생은 새로운 문제도 잘 풉니다.
- 문제 답만 외운 학생은 비슷한 문제는 맞히지만, 조금만 바뀌면 틀립니다.

### 과적합이 생기는 이유
- 데이터가 너무 적음
- 모델이 너무 복잡함
- 학습을 너무 오래 함
- 노이즈가 많은 데이터

### 방지 방법
- 더 많은 데이터 사용
- 모델 단순화
- 정규화(regularization) 적용
- 드롭아웃(dropout), 조기 종료(early stopping)
- 데이터 증강(data augmentation)

원하시면 제가 **과적합과 과소적합의 차이**도 같이 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [10]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

홍길동님입니다.


In [ ]:
messages = chat_prompt.invoke({"history": [], "question": "내 이름이 뭐라고?"})
print(llm.invoke(messages).content)

# LLM 은 응답을 statless 이기 때문에 이전 대화 기록을 전달하지 않으면 이전 대화 내용을 알 수 없습니다.
# 그렇기 때문에 LangChain 에서는 이전 대화 기록을 관리하는 Memory 를 제공합니다.

아직 말씀해주신 적이 없어서 모르겠어요.  
원하시면 제가 기억할 수 있게 이름을 알려주세요.


In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory


In [8]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이라고 하셨습니다! 맞나요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이라고 하셨습니다! 맞나요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

입력(한국어): "오늘 날씨가 정말 좋네요."  
출력(영어): The weather is really nice today.
